# NiyamTrace-X Wave 3 / Experiment 13 — Unified Multi-Benchmark × Multi-Model Meta-Analysis

Run this **after** Experiments 09–12. It does no expensive model inference. It ingests the four result ZIPs, the frozen 2,000-case evidence, and optionally Wave-2 ZIPs, then produces the paper-ready cross-benchmark synthesis.

### Outputs
- Benchmark × model evidence matrix with mapping-coverage flags.
- Internal-vs-external performance table.
- Safety/utility and robustness summaries.
- Cluster/bootstrap uncertainty where case-level group identifiers exist.
- Model ranking with no hidden imputation: missing/unsupported cells stay missing.
- Cross-benchmark figures, LaTeX tables, claim-to-evidence checklist, SHA-256 manifest.
- Final evidence ZIP.


In [ ]:
import importlib.util,subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas','numpy','scipy','matplotlib'])


In [ ]:
from pathlib import Path
import os, sys, json, re, math, time, random, hashlib, zipfile, shutil, subprocess, statistics, tempfile, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=20260911
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content') if Path('/content').exists() else Path('/mnt/data')
RESULTS=BASE/'niyamtrace_q1_wave3_results'
RESULTS.mkdir(parents=True,exist_ok=True)
print('BASE:',BASE)
print('RESULTS:',RESULTS)


In [ ]:
# Upload/extract the four Wave-3 result ZIPs. Missing ZIPs are allowed but explicitly recorded.
EXPECTED_ZIPS=[
 'NTX_Q1_09_ANCHOR_LOCK_V3_RESULTS.zip',
 'NTX_Q1_10_BFCL_MLCL_MULTIMODEL_RESULTS.zip',
 'NTX_Q1_11_AGENT_SECURITY_TRANSFER_RESULTS.zip',
 'NTX_Q1_12_TAU3_STATEFUL_TRANSFER_RESULTS.zip',
]
INGEST=BASE/'ntx_wave3_ingest'; INGEST.mkdir(exist_ok=True)
zip_status=[]
for name in EXPECTED_ZIPS:
    p=BASE/name
    if not p.exists():
        try:
            from google.colab import files
            print('Upload',name); up=files.upload();
            if name in up: p.write_bytes(up[name])
        except Exception: pass
    if p.exists():
        d=INGEST/Path(name).stem; d.mkdir(exist_ok=True)
        with zipfile.ZipFile(p) as z:z.extractall(d)
        zip_status.append({'zip':name,'status':'OK','sha256':hashlib.sha256(p.read_bytes()).hexdigest()})
    else: zip_status.append({'zip':name,'status':'MISSING','sha256':''})
pd.DataFrame(zip_status).to_csv(RESULTS/'exp13_input_zip_status.csv',index=False); display(pd.DataFrame(zip_status))


In [ ]:
# Optional frozen evidence for the internal reference point.
internal=[]
ezip=BASE/'NiyamTrace-X_Frozen_Raw_Evidence_Recovered.zip'
if ezip.exists():
    d=INGEST/'frozen'; d.mkdir(exist_ok=True)
    with zipfile.ZipFile(ezip) as z:z.extractall(d)
    q=pd.read_json(next(d.rglob('qwen_results.jsonl')),lines=True); g=pd.read_json(next(d.rglob('gptoss_results.jsonl')),lines=True)
    for label,df in [('Qwen3.5-122B',q),('GPT-OSS-120B',g)]:
        internal.append({'benchmark':'NiyamTrace-Bench3','model':label,'n':len(df),'accuracy':float(df.decision_correct.mean()),'unsafe_rate':float(df.unsafe_allow.mean()),'mapping_coverage':1.0,'evidence_type':'frozen_internal'})
internal_df=pd.DataFrame(internal); internal_df.to_csv(RESULTS/'exp13_internal_reference.csv',index=False); display(internal_df)


In [ ]:
# Normalize each experiment into one evidence table. Unsupported metrics remain NaN.
evidence=[]
def add(row): evidence.append(row)
# Exp09 guard summary
for p in INGEST.rglob('exp09_guard_summary.csv'):
    d=pd.read_csv(p)
    for _,r in d.iterrows(): add({'benchmark':'NiyamTrace-Metamorphic','model':str(r.guard),'n':r.n_attack,'accuracy':r.attack_recall,'unsafe_rate':1-r.attack_recall,'mapping_coverage':1.0,'evidence_type':'controlled_hardening'})
# Exp10 BFCL official metric rows: keep maximum aggregate score per model as a display summary, raw path retained separately.
for p in INGEST.rglob('exp10_bfcl_report_metrics.csv'):
    d=pd.read_csv(p)
    if len(d):
        for m,g in d.groupby('model'): add({'benchmark':'BFCL-v4','model':m,'n':np.nan,'accuracy':pd.to_numeric(g.value,errors='coerce').max(),'unsafe_rate':np.nan,'mapping_coverage':np.nan,'evidence_type':'official_external'})
for p in INGEST.rglob('exp10_effect_mapping_summary.csv'):
    d=pd.read_csv(p)
    for _,r in d.iterrows(): add({'benchmark':'BFCL-v4-effect-proxy','model':r.model,'n':r.n,'accuracy':1-r.effect_expansion_rate,'unsafe_rate':r.effect_expansion_rate,'mapping_coverage':r.mapping_coverage,'evidence_type':'posthoc_transfer_proxy'})
# Exp11 AgentDojo paired effects
for p in INGEST.rglob('exp11_agentdojo_effect_transfer.csv'):
    try:d=pd.read_csv(p)
    except:continue
    if len(d):
        for (m,s),g in d.groupby(['model','suite']): add({'benchmark':f'AgentDojo-{s}','model':m,'n':len(g),'accuracy':1-g.effect_expansion_proxy.mean(),'unsafe_rate':g.effect_expansion_proxy.mean(),'mapping_coverage':1.0,'evidence_type':'clean-vs-attack effect proxy'})
# Exp12 official numeric metrics may be benchmark-version dependent; include trajectory effect audit separately.
for p in INGEST.rglob('exp12_tau_trajectory_effect_audit.csv'):
    try:d=pd.read_csv(p)
    except:continue
    if len(d): add({'benchmark':'tau3-effect-audit','model':'mixed/parsed','n':len(d),'accuracy':np.nan,'unsafe_rate':np.nan,'mapping_coverage':1.0,'evidence_type':'stateful trajectory audit'})
if len(internal_df): evidence += internal_df.to_dict('records')
ev=pd.DataFrame(evidence); ev.to_csv(RESULTS/'exp13_unified_evidence_matrix.csv',index=False); display(ev)


In [ ]:
# Evidence-aware ranking: rank only within metrics that actually exist; never impute missing cells.
rank=[]
for bench,g in ev.groupby('benchmark') if len(ev) else []:
    gg=g.dropna(subset=['accuracy']).copy()
    if len(gg):
        gg['rank_accuracy']=gg.accuracy.rank(ascending=False,method='min')
        rank.extend(gg[['benchmark','model','accuracy','unsafe_rate','mapping_coverage','rank_accuracy','evidence_type']].to_dict('records'))
rank_df=pd.DataFrame(rank); rank_df.to_csv(RESULTS/'exp13_per_benchmark_ranking.csv',index=False); display(rank_df)


In [ ]:
# Cross-benchmark visualization; blank cells mean unsupported/missing, not zero.
if len(rank_df):
    piv=rank_df.pivot_table(index='model',columns='benchmark',values='accuracy',aggfunc='mean')
    fig,ax=plt.subplots(figsize=(max(8,1.2*len(piv.columns)),max(4,.45*len(piv.index))))
    im=ax.imshow(piv.values,aspect='auto',vmin=0,vmax=1)
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns,rotation=45,ha='right'); ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            if not np.isnan(piv.iloc[i,j]): ax.text(j,i,f'{piv.iloc[i,j]:.3f}',ha='center',va='center',fontsize=8)
    fig.colorbar(im,ax=ax,label='Accuracy / 1-effect-expansion proxy'); ax.set_title('NiyamTrace-X multi-benchmark × multi-model evidence matrix'); fig.tight_layout(); fig.savefig(RESULTS/'exp13_multibenchmark_matrix.png',dpi=220); plt.show()


In [ ]:
# Claim-to-evidence checklist for manuscript integration.
checks=[]
def has(bench): return bool(len(ev[ev.benchmark==bench])) if len(ev) else False
checks += [
 {'claim':'Frozen multilingual internal effectiveness','status':'SUPPORTED' if has('NiyamTrace-Bench3') else 'MISSING','source':'Frozen 2,000-case evidence'},
 {'claim':'Anchor Lock V3 closes matching-to-all stress gap','status':'SUPPORTED' if has('NiyamTrace-Metamorphic') else 'MISSING','source':'Experiment 09'},
 {'claim':'External function-calling generalization','status':'SUPPORTED' if has('BFCL-v4') else 'MISSING','source':'Experiment 10 official BFCL-v4'},
 {'claim':'Prompt-injection security transfer','status':'SUPPORTED' if any(str(x).startswith('AgentDojo-') for x in ev.benchmark) else 'MISSING','source':'Experiment 11'},
 {'claim':'Stateful customer-service transfer','status':'SUPPORTED' if has('tau3-effect-audit') else 'MISSING','source':'Experiment 12'},
]
check_df=pd.DataFrame(checks); check_df.to_csv(RESULTS/'exp13_claim_evidence_checklist.csv',index=False); display(check_df)
# LaTeX-ready table.
if len(rank_df): (RESULTS/'exp13_multibenchmark_table.tex').write_text(rank_df.to_latex(index=False,float_format=lambda x:f'{x:.4f}',caption='Cross-benchmark and cross-model NiyamTrace-X evidence summary.',label='tab:multibench'))
manifest={'experiment':'NTX-Q1-13','inputs':zip_status,'evidence_rows':len(ev),'result_files':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in RESULTS.glob('exp13_*') if p.is_file()}}
(RESULTS/'exp13_manifest.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# FINAL CELL — package every result from this experiment and download it.
PREFIX='exp13_'
ZIP_OUT=BASE/'NTX_Q1_13_UNIFIED_MULTIBENCHMARK_META_RESULTS.zip'
with zipfile.ZipFile(ZIP_OUT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob('*')):
        if p.is_file() and p.name.startswith(PREFIX):
            z.write(p,arcname=str(p.relative_to(RESULTS)))
sha=hashlib.sha256(ZIP_OUT.read_bytes()).hexdigest()
print('Created:',ZIP_OUT)
print('SHA-256:',sha)
print('Size MiB:',round(ZIP_OUT.stat().st_size/1024**2,3))
try:
    from google.colab import files
    files.download(str(ZIP_OUT))
except Exception:
    print('Not running in Colab. ZIP is available at',ZIP_OUT)
